In [4]:
import pandas as pd

big_matrix = pd.read_csv(
    "/home/sulay/deep-linear-bandits/kuairec/data/big_matrix.csv",
    usecols=[
        'user_id',
        'video_id',
        'watch_ratio'
    ]
)

big_matrix

,user_id,video_id,watch_ratio
0,0,3649,1.273397
1,0,9598,1.244082
2,0,5262,0.107613
3,0,1963,0.089885
4,0,8234,0.078000
...,...,...,...
12530801,7175,1281,0.247241
12530802,7175,3407,0.576526
12530803,7175,10360,0.340597
12530804,7175,10360,0.913400


In [5]:
# Duplicates are known to exist for user-video pairs (multiple watch_ratios observed); would need to take e.g. max of their
# watch ratios, but can be avoided by just applying the binary signal early and removing duplicates for the same effect

# d = big_matrix.duplicated(subset=['user_id', 'video_id'], keep=False)
# big_matrix[d].sort_values(by=['user_id', 'video_id', 'watch_ratio'])

big_matrix = big_matrix[big_matrix.watch_ratio >= 2.0].drop(columns=['watch_ratio'])

big_matrix

,user_id,video_id
7,0,6812
11,0,5274
12,0,179
17,0,171
23,0,211
...,...,...
12530735,7175,6502
12530749,7175,7951
12530759,7175,2970
12530767,7175,8898


In [6]:
big_matrix = big_matrix.drop_duplicates()

big_matrix

,user_id,video_id
7,0,6812
11,0,5274
12,0,179
17,0,171
23,0,211
...,...,...
12530735,7175,6502
12530749,7175,7951
12530759,7175,2970
12530767,7175,8898


In [7]:
big_matrix.groupby(['user_id']).count().sort_values(by=['video_id']) # note that if I do a per-user split some users have little data

,video_id
user_id,
1282,1
735,1
5757,2
4427,2
6384,3
...,...
5684,683
2465,688
933,705


In [8]:
# Was just verifying that NaNs are removed (they are)
# import numpy as np
# test = pd.DataFrame([np.nan, np.nan, 5, 1], columns=['foo'])
# test['foo'] >= 2

The bandit assumes that it's operating within a stationary (non-drifting relative to time) embedding window  
Obviously for evaluation it doesn't make sense (and is not feasible) to also have complete consideration of non-stationary wrt time  
The two-tower model however expects to be non-stationary long term & will be retrained, so it can make sense for it to be time-facing  

(per-user or global) temporal splits may be ideal  

the two-tower can be expected to change over time, and goal of feedback loop work will be to ensure that the two-tower doesn't remain stationary  
new data should come into the recommender: the current (state of the / spread of users and items in the) embedding space that two-tower has converged to should do a good job at approximating
the current state of user-item interaction patterns  
given new information, ideally two-tower should aim to converge to a new ideal embedding space - this requires separate work that looks at whether two-tower (when set up correctly) can converge to new embedding spaces efficiently (likely not, due to policy stagnation studies)

questions at the moment:
- two-tower aims to place users near items that they like and vice-versa; how effectively does it cater for the fact that user interests may reflect disjoint clusters across that embedding space? it seems like it will be limited to placing the user in an "average" between those clusters rather than ensuring e.g. a well-represented item space with users potentially mapping to multiple clusters on that item space; perhaps with high-dimensionality though this is fine, as in reality there will be heavy relationships between item clusters that can be captured - i.e. if this is a meaningful relationship, we can assume similarity in the items across dimensions whether that is actually meaningful relative to the item's properties or just useful in terms of embedding items similarly that we can know users like similarly - this of course needs future work as well
- do the linear bandits help to mitigate this effect at all - can they learn disjoint clusters (i.e. if a certain class of users are found to like two classes of items that are currently not placed closely in the embedding space)? (we can see; and the bandit policy can ensure interactions that are fed back to the model for newly learning similarity between those clusters, however this also requires some notion of bandit memory after the feedback update on the model i.e. warming the bandits relative to changing embeddings, which introduces nonstationarity issues + optimising for a new environment/state especially over longer periods is likely suited for RL)
- visualising the way that the linear bandit will actually act on user-item embeddings and learn a linear relationship within the embedding space

the bandit (on small matrix, out of time order) operates over an overlapping time span with the two-tower training data; it's assuming a single stationary environment for context vectors across the entire span of the experiment (feasible across entire user-item space as it's the span of months, not years)  

this is generally feasible, as you can assume within this time span that the model's understanding of the landscape of users and items is generally good

the supervised (two-tower) side is learning an effective representation of an assumed stationary environment (representing this time span) of users and items across the time period, with lots of historical user-item interaction data to inform this (the observational approach); the bandits take an interventional approach (in intervening to gain more information, learn a better model; not from the side of shifting the user preferences over time to create concept/environment drift) over the same assumed stationary environment to optimise adaptively in real-time to what users actually like over the same environment, with more expressive power than simple dot product (since the context vector includes element wise product)

in reality the bandits would be temporally ahead of the two-tower and pulling it towards a new learning of user-item embeddings (potentially), but more realistically given two-tower update delay + likely taking a long time to shift its embedding space + assuming that the environment does not change massively across all users and items relative to what has generally been seen historically + using the stationarity assumption central to this project, then it is just learning a more effective understanding of user-item relationships within a fixed stationary environment using adaptive feedback via exploration (and also ensuring that users have a better experience via exploration too, challenging the static nature of the model)

this makes it more of a problem of information deficit on the current nature of user-item interactions (especially for user or item clusters with low and sparse data) and/or just model deficit due to two-tower and the supervised/interaction history approach  

arguably does not need a temporal split on two-tower to do this; however alternatively might still be a wise choice -> showing that the two-tower embedding space effectively represents future interactions? however this is in contrast to it knowing its own current (stationary) environment which can be tested more rigorously via the random sampling and is better suited for gauging its ability to currently know its environment before the bandit then acts on top -> I do think it is better to unify the stationary assumption here across two tower and the bandit

the bandit for example is going to show interactions that are between known two-tower interactions (and wants to act irrespective of time) but if two-tower is optimised relative to a concept drift across the timespan of its dataset, aiming to favour an understanding of the environment nearer its end, the bandit's performance on an assumed stationary environment is compromised -> random splits ensure that the two-tower is learning a time-independent approximate understanding of the assumed stationary environment

aim for 80/20 split -> 80% training, 20% validation; as it's a multistage model the testing is done at small matrix level

In [9]:
big_matrix

,user_id,video_id
7,0,6812
11,0,5274
12,0,179
17,0,171
23,0,211
...,...,...
12530735,7175,6502
12530749,7175,7951
12530759,7175,2970
12530767,7175,8898


aim to:
- split into 80/20 (on users; cold-start analysis should be separate to scope of this proj) accounting for users with not enough samples
- check small matrix users and videos completely in big matrix (should be)
- gather additional input data (features) for the model; if some are unknown then handle this as standard for categorical vars with unknowns?
- finalise dataset & dataloader logic; use for two-tower model

In [10]:
big_matrix

,user_id,video_id
7,0,6812
11,0,5274
12,0,179
17,0,171
23,0,211
...,...,...
12530735,7175,6502
12530749,7175,7951
12530759,7175,2970
12530767,7175,8898


In [12]:
small_matrix = pd.read_csv(
    "/home/sulay/deep-linear-bandits/kuairec/data/small_matrix.csv",
    usecols=[
        'user_id',
        'video_id',
        'watch_ratio'
    ]
)

small_matrix

,user_id,video_id,watch_ratio
0,14,148,0.722103
1,14,183,1.907377
2,14,3649,2.063311
3,14,5262,0.566388
4,14,8234,0.418364
...,...,...,...
4676565,7162,2267,2.178160
4676566,7162,2065,1.964562
4676567,7162,1296,0.839960
4676568,7162,4822,0.486148


In [ ]:
small_matrix['user_id'].unique()

array([  14,   19,   21, ..., 7153, 7159, 7162], shape=(1411,))

In [21]:
big_matrix.groupby(['user_id']).count()

,video_id
user_id,
0,258
1,166
2,38
3,163
4,51
...,...
7171,134
7172,271
7173,95


In [22]:
big_matrix.groupby(['user_id']).count().iloc[small_matrix['user_id'].unique()].sort_values(by='video_id')

,video_id
user_id,
735,1
1282,1
4427,2
2787,3
4528,3
...,...
5525,445
5657,459
6008,461


In [ ]:
big_matrix

,user_id,video_id
7,0,6812
11,0,5274
12,0,179
17,0,171
23,0,211
...,...,...
12530735,7175,6502
12530749,7175,7951
12530759,7175,2970
12530767,7175,8898


In [34]:
big_matrix.groupby(['user_id']).count().sort_values(by='video_id')

,video_id
user_id,
1282,1
735,1
5757,2
4427,2
6384,3
...,...
5684,683
2465,688
933,705


In [ ]:
low_interactions = big_matrix.groupby(['user_id']).count() < 5

low_interactions.columns = ['count']

low_interactions

,count
user_id,
0,False
1,False
2,False
3,False
4,False
...,...
7171,False
7172,False
7173,False


In [121]:
low = big_matrix.groupby(['user_id'])['user_id'].transform('count') < 5

low

7           False
11          False
12          False
17          False
23          False
            ...  
12530735    False
12530749    False
12530759    False
12530767    False
12530794    False
Name: user_id, Length: 846414, dtype: bool

In [122]:
big_matrix[low]

,user_id,video_id
138390,75,886
138416,75,5946
138497,75,6463
138514,75,3400
254771,137,9551
...,...,...
11166663,6384,2264
11627800,6658,4374
11628117,6658,10653
11628144,6658,3400


In [123]:
big_matrix[low]

,user_id,video_id
138390,75,886
138416,75,5946
138497,75,6463
138514,75,3400
254771,137,9551
...,...,...
11166663,6384,2264
11627800,6658,4374
11628117,6658,10653
11628144,6658,3400


In [125]:
from sklearn.model_selection import train_test_split

a, b = train_test_split(
    big_matrix[~low],
    train_size=0.8,
    stratify=big_matrix[~low]['user_id']
)

a

,user_id,video_id
8348004,4742,1288
3932202,2252,7084
7214842,4108,7103
2849871,1620,7358
7826670,4448,9106
...,...,...
8590038,4884,672
132026,71,5624
9856141,5618,853
2093118,1192,8488


In [126]:
b

,user_id,video_id
9074814,5157,4166
3362233,1921,10382
658358,362,5794
244831,128,3742
8031759,4563,10318
...,...,...
6172202,3529,8529
8583655,4880,435
788999,433,1336
6839910,3901,8302


In [129]:
pd.concat([a, big_matrix[low]])

,user_id,video_id
8348004,4742,1288
3932202,2252,7084
7214842,4108,7103
2849871,1620,7358
7826670,4448,9106
...,...,...
11166663,6384,2264
11627800,6658,4374
11628117,6658,10653
11628144,6658,3400
